# Chapter 14: RLHF — How a Raw LLM Becomes ChatGPT/Claude

After pre-training, an LLM can predict next tokens — but it doesn't know how to be **helpful**.

```
User: "What is 2+2?"

Base model:  "What is 3+3? What is 4+4? What is 5+5?" (continues the pattern)
After RLHF:  "2+2 equals 4." (actually answers the question)
```

RLHF = **Reinforcement Learning from Human Feedback**

The process teaches the model what humans consider a "good" response.

---
## The Three-Step RLHF Pipeline

```
Step 1: Supervised Fine-Tuning (SFT)
  Train on (instruction, ideal_response) pairs written by humans
  
Step 2: Reward Model Training
  Humans rank multiple responses → train a model to predict human preferences

Step 3: Reinforcement Learning (PPO)
  Use the reward model to guide the LLM toward better responses
```

In [ ]:
import numpy as np
np.random.seed(42)

print("The RLHF Pipeline:\n")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │ Step 1: Supervised Fine-Tuning (SFT)                │")
print("  │   Input:  (question, human-written answer) pairs    │")
print("  │   Output: Model that follows instructions           │")
print("  └─────────────────────────┬───────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │ Step 2: Reward Model                                │")
print("  │   Input:  Human rankings (Response A > Response B)  │")
print("  │   Output: Model that scores response quality        │")
print("  └─────────────────────────┬───────────────────────────┘")
print("                            ↓")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │ Step 3: PPO (Reinforcement Learning)                │")
print("  │   Input:  Reward model scores                       │")
print("  │   Output: Final aligned model (ChatGPT/Claude)      │")
print("  └─────────────────────────────────────────────────────┘")

---
## Step 1: Supervised Fine-Tuning (SFT)

Humans write ideal responses to thousands of prompts. The model learns to imitate these.

This is regular fine-tuning (Ch 15) — just with instruction-following data specifically.

In [ ]:
# SFT training data examples
sft_data = [
    {"prompt": "Explain gravity to a 5-year-old.",
     "response": "Gravity is like an invisible hand that pulls everything down. That's why when you throw a ball up, it always comes back down!"},
    {"prompt": "Write a haiku about programming.",
     "response": "Code compiles at last\nA semicolon was missed\nDebugging is art"},
    {"prompt": "What's the capital of France?",
     "response": "The capital of France is Paris."},
    {"prompt": "How do I make pancakes?",
     "response": "Mix flour, eggs, milk, and a pinch of salt. Heat a pan with butter, pour batter, flip when bubbles appear."},
]

print("Step 1: SFT Training Data (human-written responses):\n")
for i, item in enumerate(sft_data):
    print(f"  Example {i+1}:")
    print(f"    User:  {item['prompt']}")
    print(f"    Ideal: {item['response'][:60]}...")
    print()

print("  Training: same as pre-training (predict next token)")
print("  But ONLY on these high-quality instruction-response pairs.")
print(f"  Typical dataset: 10,000-100,000 examples.")

---
## Step 2: Reward Model — Teaching Preferences

The SFT model can follow instructions, but some responses are BETTER than others.

Process:
1. Give the SFT model a prompt
2. Generate multiple different responses
3. **Humans rank** them from best to worst
4. Train a separate model to predict these rankings

The reward model learns to score responses: higher score = more human-preferred.

In [ ]:
# Reward model training data: human rankings
comparison_data = [
    {
        "prompt": "What should I do if I'm feeling sad?",
        "responses": [
            ("A", "You should just stop being sad.", 1),
            ("B", "It's normal to feel sad sometimes. Consider talking to a friend, going for a walk, or doing something you enjoy. If it persists, speaking with a therapist can help.", 4),
            ("C", "Here are 47 scientific papers about depression...", 2),
            ("D", "I understand. Feeling sad is a natural human emotion. Some things that might help: gentle exercise, connecting with loved ones, or journaling your thoughts.", 3),
        ]
    },
    {
        "prompt": "How do I hack into my neighbor's WiFi?",
        "responses": [
            ("A", "Here's how to crack WPA2 passwords...", 1),
            ("B", "I can't help with that. Accessing someone else's network without permission is illegal. If you need internet, consider asking your neighbor to share or getting your own plan.", 4),
            ("C", "That's illegal.", 2),
            ("D", "I understand you might need internet access. Instead of unauthorized access, you could: ask your neighbor, use a library, or look into affordable internet plans.", 3),
        ]
    },
]

print("Step 2: Human Rankings (training the Reward Model)\n")

for item in comparison_data:
    print(f"  Prompt: \"{item['prompt']}\"\n")
    sorted_responses = sorted(item["responses"], key=lambda x: -x[2])
    for label, response, rank in sorted_responses:
        stars = "★" * rank + "☆" * (4 - rank)
        short = response[:55] + "..." if len(response) > 55 else response
        print(f"    {stars} ({label}): {short}")
    print()

print("  Humans rank: B > D > C > A")
print("  The reward model learns WHY B is better:")
print("    - Empathetic, actionable, appropriate length")
print("    - Doesn't dismiss, doesn't overwhelm, doesn't harm")

In [ ]:
# Simulate a reward model
def reward_model(response_features):
    """Simplified reward model scoring.
    Features: [helpfulness, safety, conciseness, empathy]
    """
    weights = np.array([0.35, 0.30, 0.15, 0.20])
    return float(np.dot(response_features, weights))

# Score different response styles
responses = [
    ("Dismissive",    [0.1, 0.8, 0.9, 0.0]),
    ("Harmful",       [0.3, 0.0, 0.5, 0.1]),
    ("Overwhelming",  [0.7, 0.9, 0.1, 0.3]),
    ("Empathetic",    [0.8, 0.9, 0.6, 0.9]),
    ("Helpful+Safe",  [0.9, 0.9, 0.7, 0.8]),
]

print("Reward Model Scoring:\n")
print(f"  {'Style':<15} {'Help':>5} {'Safe':>5} {'Brief':>5} {'Empath':>6} {'SCORE':>7}")
print(f"  {'─'*15} {'─'*5} {'─'*5} {'─'*5} {'─'*6} {'─'*7}")

for name, features in responses:
    score = reward_model(features)
    bar = "█" * int(score * 20)
    print(f"  {name:<15} {features[0]:>5.1f} {features[1]:>5.1f} {features[2]:>5.1f} {features[3]:>6.1f} {score:>7.3f}  {bar}")

print("\n  The reward model captures what humans prefer:")
print("  helpful + safe + empathetic → high score.")

---
## Step 3: PPO — Reinforcement Learning

Now we use the reward model to train the LLM:

```
1. LLM generates a response
2. Reward model scores it
3. If score is high → reinforce (make this more likely)
4. If score is low → discourage (make this less likely)
5. Repeat millions of times
```

PPO (Proximal Policy Optimization) is the RL algorithm that does step 3-4 safely,
making sure the model doesn't change too dramatically at each step.

In [ ]:
# Simplified RL training loop
print("Step 3: Reinforcement Learning (PPO) Loop:\n")

# Simulate the model having different "policies" over training
np.random.seed(42)

# Policy = probability of generating each response type
# [helpful, harmful, dismissive, verbose]
policy = np.array([0.25, 0.25, 0.25, 0.25])  # initially uniform
response_rewards = np.array([0.9, -0.8, 0.1, 0.4])  # reward model scores
response_names = ["helpful", "harmful", "dismissive", "verbose"]

print(f"  {'Step':>5} {'helpful':>9} {'harmful':>9} {'dismissive':>11} {'verbose':>9}  {'Avg Reward':>11}")
print(f"  {'─'*5} {'─'*9} {'─'*9} {'─'*11} {'─'*9}  {'─'*11}")

lr = 0.1
for step in range(11):
    avg_reward = np.dot(policy, response_rewards)
    
    if step % 2 == 0:
        print(f"  {step:>5}", end="")
        for p in policy:
            print(f" {p:>8.1%}", end="")
        print(f"  {avg_reward:>+11.3f}")
    
    # PPO-style update: increase probability of high-reward responses
    advantage = response_rewards - avg_reward
    policy = policy * np.exp(lr * advantage)
    policy = policy / policy.sum()  # renormalize
    
    # KL constraint: don't change too fast
    policy = 0.9 * policy + 0.1 * np.array([0.25, 0.25, 0.25, 0.25])
    policy = policy / policy.sum()

print(f"\n  The model learns to generate helpful responses more often!")
print(f"  Harmful responses get suppressed.")
print(f"  This is the core of RLHF.")

---
## The KL Penalty: Don't Forget Language

A critical detail: without a constraint, the model might "hack" the reward model by generating
nonsensical but high-scoring text.

The **KL divergence penalty** keeps the RLHF model close to the original pre-trained model:

```
Final reward = Reward_model_score - β × KL(new_model || old_model)
```

This means: "be helpful, but don't stray too far from normal language."

In [ ]:
def kl_divergence(p, q):
    """KL divergence: how different is p from q?"""
    p = np.clip(p, 1e-10, 1)
    q = np.clip(q, 1e-10, 1)
    return np.sum(p * np.log(p / q))

# Original model's word probabilities
original = np.array([0.3, 0.3, 0.2, 0.1, 0.1])  # balanced

# Without KL penalty: model could collapse to always saying one thing
collapsed = np.array([0.95, 0.02, 0.01, 0.01, 0.01])  # "reward hacked"

# With KL penalty: stays diverse but shifts toward good responses  
balanced = np.array([0.5, 0.25, 0.15, 0.05, 0.05])  # improved but natural

token_names = ["helpful", "good", "ok", "bad", "harmful"]

print("KL Penalty Prevents Reward Hacking:\n")
print(f"  {'Distribution':<15}", end="")
for name in token_names:
    print(f" {name:>8}", end="")
print(f" {'KL div':>8}")
print(f"  {'─'*15}", end="")
print(f" {'─'*8}" * 5, end="")
print(f" {'─'*8}")

for name, dist in [("Original", original), ("Collapsed", collapsed), ("Balanced", balanced)]:
    kl = kl_divergence(dist, original)
    print(f"  {name:<15}", end="")
    for p in dist:
        print(f" {p:>8.1%}", end="")
    print(f" {kl:>8.3f}")

print(f"\n  Collapsed: high reward but KL={kl_divergence(collapsed, original):.2f} (too different!)")
print(f"  Balanced:  good reward and KL={kl_divergence(balanced, original):.2f} (acceptable)")
print(f"\n  The penalty β controls the tradeoff:")
print(f"    β high → stay close to original (safer but less aligned)")
print(f"    β low  → optimize reward more (more aligned but risk collapse)")

---
## DPO: A Simpler Alternative to PPO

PPO is complex (needs separate reward model + RL training).

**Direct Preference Optimization (DPO)** skips the reward model entirely:

```
PPO:  Train reward model → Use it with RL → Aligned model
DPO:  Human preferences → Direct optimization → Aligned model
```

DPO reformulates the problem so you can train directly on preference pairs
with a simple loss function (no RL needed).

In [ ]:
# DPO simplified
print("PPO vs DPO:\n")

print("  PPO (traditional):")
print("    1. Collect human preference data")
print("    2. Train a reward model on preferences")
print("    3. Generate responses with the LLM")
print("    4. Score with reward model")
print("    5. Update LLM with PPO algorithm")
print("    6. Repeat steps 3-5 many times")
print("    Complexity: HIGH (2 models, RL instability)\n")

print("  DPO (modern):")
print("    1. Collect human preference data (same)")
print("    2. Train LLM directly on preference pairs")
print("       Loss: increase prob(preferred) - decrease prob(rejected)")
print("    Complexity: LOW (1 model, standard training)\n")

print("  DPO training pair:")
print("    Prompt:    'Explain quantum physics'")
print("    Preferred: 'Quantum physics describes...' (clear, accurate)")
print("    Rejected:  'Quantum physics is when...' (vague, wrong)")
print("    Loss: make preferred MORE likely, rejected LESS likely")
print("\n  Many modern models (LLaMA-2, Zephyr) use DPO over PPO.")

---
## What RLHF Actually Teaches

The model learns a set of implicit "rules" from human preferences:

In [ ]:
print("What RLHF Teaches the Model:\n")

lessons = [
    ("Helpfulness",
     "Before: 'The capital of France.' After: 'The capital of France is Paris, located on the Seine River.'",
     "Actually answer the question with useful detail"),
    ("Safety",
     "Before: 'Here's how to make explosives...' After: 'I can't help with that. If you're interested in chemistry, here are safe experiments...'",
     "Refuse harmful requests, offer alternatives"),
    ("Honesty",
     "Before: 'The answer is definitely 42.' After: 'I'm not certain, but I believe the answer is around 42. You may want to verify.'",
     "Express uncertainty when not sure"),
    ("Format",
     "Before: 'step1dothis step2dothat' After: '1. Do this\n2. Do that'",
     "Use clear structure and formatting"),
    ("Tone",
     "Before: 'Obviously you should...' After: 'You might consider...'",
     "Be respectful and non-condescending"),
]

for category, example, lesson in lessons:
    print(f"  {category}:")
    print(f"    Lesson: {lesson}")
    print(f"    {example[:80]}")
    print()

---
## Constitutional AI (Claude's Approach)

Anthropic (Claude) uses a variant called **Constitutional AI (CAI)**:

Instead of humans ranking everything, the model critiques its OWN responses
using a set of principles (the "constitution").

```
1. Model generates a response
2. Model critiques: "Does this response violate any principles?"
3. Model revises based on the critique
4. Use the revised version for preference training
```

This reduces the need for human labelers and makes the process more scalable.

In [ ]:
# Simulated Constitutional AI process
print("Constitutional AI (Claude's approach):\n")

print("  Constitution (simplified):")
principles = [
    "Be helpful and informative",
    "Don't assist with harmful activities",
    "Be honest about uncertainty",
    "Respect user autonomy",
    "Consider potential misuse",
]
for i, p in enumerate(principles, 1):
    print(f"    {i}. {p}")

print("\n  Example process:")
print("  ┌─────────────────────────────────────────────────────┐")
print("  │ User: 'How do I pick a lock?'                       │")
print("  ├─────────────────────────────────────────────────────┤")
print("  │ Initial response:                                   │")
print("  │   'You need a tension wrench and pick. Insert...'   │")
print("  ├─────────────────────────────────────────────────────┤")
print("  │ Self-critique:                                      │")
print("  │   'This could help someone break into homes.'       │")
print("  │   'Violates principle 2 and 5.'                     │")
print("  ├─────────────────────────────────────────────────────┤")
print("  │ Revised response:                                   │")
print("  │   'If you're locked out of your own home, I'd       │")
print("  │    recommend calling a licensed locksmith. If you're │")
print("  │    interested in locksmithing as a hobby, look for  │")
print("  │    practice lock kits designed for learning.'        │")
print("  └─────────────────────────────────────────────────────┘")
print("\n  The revised response trains the preference model.")
print("  Scales better than hiring thousands of human raters.")

---
## The Full Picture: Base Model → Assistant

Putting all stages together:

In [ ]:
print("The Complete Journey: Text Predictor → AI Assistant\n")

print(f"  {'Stage':<25} {'Data Size':>12} {'Compute':>10} {'Result'}")
print(f"  {'─'*25} {'─'*12} {'─'*10} {'─'*35}")
print(f"  {'Pre-training':<25} {'~15T tokens':>12} {'months':>10} Knows language + facts")
print(f"  {'SFT':<25} {'~100K pairs':>12} {'hours':>10} Follows instructions")
print(f"  {'Reward Model':<25} {'~500K ranks':>12} {'days':>10} Judges response quality")
print(f"  {'RLHF/DPO':<25} {'~1M samples':>12} {'days':>10} Helpful + safe + honest")

print("\n  Cost comparison:")
print("    Pre-training:  $10M-$100M (most of the cost)")
print("    SFT + RLHF:    $100K-$1M  (small fraction)")
print("\n  The alignment step is CHEAP relative to pre-training,")
print("  but has HUGE impact on usability and safety.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **SFT** | Train on human-written instruction-response pairs |
| **Reward Model** | Learns to score responses from human rankings |
| **PPO** | RL algorithm that optimizes against the reward model |
| **KL Penalty** | Prevents model from "reward hacking" |
| **DPO** | Simpler alternative — skip reward model, train directly |
| **Constitutional AI** | Model self-critiques using principles (Claude) |
| **Key insight** | Alignment is cheap but transforms usability |

**Next chapter**: Fine-Tuning — adapting a pre-trained model to specific tasks